# First look — Bike Share Toronto ridership + stations

Goal of this notebook: pull the latest full-year ridership zip and the station-info GBFS feed, unpack, peek at schema, and compute a first-pass OD matrix.

No conclusions yet — just ground truth the data shape.

In [1]:
import sys, zipfile, json
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / 'src'))

import pandas as pd
import duckdb
from bikeshare.fetch import (
    package_resources, download, save_manifest,
    RIDERSHIP_PACKAGE, STATION_PACKAGE,
)

DATA_RAW = Path.cwd().parent / 'data' / 'raw'
DATA_RAW.mkdir(parents=True, exist_ok=True)

## 1. Discover resources

In [2]:
ridership_res = package_resources(RIDERSHIP_PACKAGE)
station_res = package_resources(STATION_PACKAGE)
save_manifest(RIDERSHIP_PACKAGE, ridership_res, DATA_RAW)
save_manifest(STATION_PACKAGE, station_res, DATA_RAW)
pd.DataFrame([r.__dict__ for r in ridership_res])[['name', 'format', 'last_modified']]

,name,format,last_modified
0,bikeshare-ridership-readme,xlsx,2022-04-12T14:38:55
1,bikeshare-ridership-2014-2015,xlsx,2022-04-11T15:42:55
2,bikeshare-ridership-2016,xlsx,2022-04-11T15:48:24
3,bikeshare-ridership-2017.zip,zip,2022-04-11T15:48:57
4,bikeshare-ridership-2018.zip,zip,2022-04-11T15:52:05
5,bikeshare-ridership-2019.zip,zip,2022-04-11T15:55:06
6,bikeshare-ridership-2020.zip,zip,2022-04-11T15:55:11
7,bikeshare-ridership-2021.zip,zip,2022-04-11T16:01:24
8,bikeshare-ridership-2022.zip,zip,2023-01-18T17:36:31
9,bikeshare-ridership-2023.zip,zip,2024-01-09T17:24:07


In [3]:
# Pick the most recent *complete* year. 2026 is in progress, so default to 2025.
target_year = '2025'
ridership_zip_res = next(r for r in ridership_res if target_year in r.name and r.format == 'zip')
ridership_zip_res

Resource(id='dedb033b-b6e9-45d1-baa4-836d1e0c7f1b', name='bikeshare-ridership-2025.zip', format='zip', url='https://opendata.toronto.ca/toronto.parking.authority/bike-share-toronto-ridership-data/bikeshare-ridership-2025.zip', last_modified='2026-04-15T18:58:36')

## 2. Download + unpack ridership

In [4]:
zip_path = download(ridership_zip_res, DATA_RAW / 'ridership')
unpack_dir = DATA_RAW / 'ridership' / target_year
unpack_dir.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(zip_path) as zf:
    zf.extractall(unpack_dir)
sorted(p.name for p in unpack_dir.glob('**/*') if p.is_file())[:20]

['bikeshare_2025_01.csv',
 'bikeshare_2025_02.csv',
 'bikeshare_2025_03.csv',
 'bikeshare_2025_04.csv',
 'bikeshare_2025_05.csv',
 'bikeshare_2025_06.csv',
 'bikeshare_2025_07.csv',
 'bikeshare_2025_08.csv',
 'bikeshare_2025_09.csv',
 'bikeshare_2025_10.csv',
 'bikeshare_2025_11.csv',
 'bikeshare_2025_12.csv']

In [5]:
csvs = sorted(unpack_dir.glob('**/*.csv'))
f'{len(csvs)} csv files; first = {csvs[0].name if csvs else None}'

'12 csv files; first = bikeshare_2025_01.csv'

In [6]:
# Peek schema from the first CSV. Toronto ridership CSVs are occasionally latin-1.
sample = csvs[0]
try:
    df_head = pd.read_csv(sample, nrows=5)
except UnicodeDecodeError:
    df_head = pd.read_csv(sample, nrows=5, encoding='latin-1')
df_head

,Trip_Id,Trip_Duration,Start_Station_Id,Start_Time,Start_Station_Name,End_Station_Id,End_Time,End_Station_Name,Bike_Id,User_Type,Bike_Model
0,34637659,193,7192,2025-01-01 00:02:15,Harbord St / Clinton St,7155,2025-01-01 00:05:28,Harbord St / Clinton St,4422,Member,ICONIC
1,34637678,464,7508,2025-01-01 00:14:07,Berkeley St / Dundas St E - SMART,7042,2025-01-01 00:21:51,Berkeley St / Dundas St E - SMART,4618,Member,ICONIC
2,34637663,770,7094,2025-01-01 00:07:21,14 Arundel Ave,7777,2025-01-01 00:20:11,14 Arundel Ave,7975,Member,EFIT G5
3,34637679,559,7032,2025-01-01 00:14:09,Augusta Ave / Dundas St W,7541,2025-01-01 00:23:28,Augusta Ave / Dundas St W,3179,Member,ICONIC
4,34637658,116,7017,2025-01-01 00:02:13,Widmer St / Adelaide St W - SMART,7102,2025-01-01 00:04:09,Widmer St / Adelaide St W - SMART,3177,Member,ICONIC


## 3. Load stations (GBFS station_information.json)

In [7]:
# The station package exposes a GBFS feed. `bike-share-json` is the top-level feed discovery doc.
import requests
gbfs_root = next(r for r in station_res if r.name == 'bike-share-json')
gbfs_root_url = gbfs_root.url
gbfs_doc = requests.get(gbfs_root_url, timeout=30).json()
feeds = gbfs_doc['data']['en']['feeds']
{f['name']: f['url'] for f in feeds}

{'system regions': 'https://tor.publicbikesystem.net/ube/gbfs/v1/en/system_regions',
 'system_information': 'https://tor.publicbikesystem.net/ube/gbfs/v1/en/system_information',
 'station_information': 'https://tor.publicbikesystem.net/ube/gbfs/v1/en/station_information',
 'station_status': 'https://tor.publicbikesystem.net/ube/gbfs/v1/en/station_status',
 'system_pricing_plans': 'https://tor.publicbikesystem.net/ube/gbfs/v1/en/system_pricing_plans'}

In [8]:
station_info_url = next(f['url'] for f in feeds if f['name'] == 'station_information')
stations = pd.DataFrame(requests.get(station_info_url, timeout=30).json()['data']['stations'])
# Keep flat scalar columns only — GBFS has nested struct/list fields (e.g. rental_uris) that parquet can't write.
flat_cols = [c for c in ['station_id', 'name', 'lat', 'lon', 'capacity', 'physical_configuration',
                          'address', 'cross_street', 'post_code', 'is_charging_station', 'rental_methods']
             if c in stations.columns]
stations_flat = stations[flat_cols].copy()
if 'rental_methods' in stations_flat:
    stations_flat['rental_methods'] = stations_flat['rental_methods'].map(
        lambda v: ','.join(v) if isinstance(v, list) else v
    )
stations_flat.to_parquet(DATA_RAW / 'stations.parquet')
stations_flat[['station_id', 'name', 'lat', 'lon', 'capacity']].head()

,station_id,name,lat,lon,capacity
0,7000,Fort York Blvd / Capreol Ct,43.639832,-79.395954,47
1,7001,Wellesley Station Green P,43.664964,-79.383550,23
2,7002,St. George St / Bloor St W,43.667131,-79.399555,19
3,7003,Madison Ave / Bloor St W,43.667018,-79.402796,15
4,7005,King St W / York St,43.648001,-79.383177,23


In [9]:
len(stations), stations[['lat', 'lon']].describe()

(1030,
                lat          lon
 count  1030.000000  1030.000000
 mean     43.682304   -79.387240
 std       0.043604     0.074652
 min      43.588077   -79.603464
 25%      43.650259   -79.431530
 50%      43.667104   -79.391471
 75%      43.706376   -79.352142
 max      43.812642   -79.123184)

## 4. First-pass OD matrix for one month

Use DuckDB to scan the first CSV without materializing the whole year in pandas. Column names vary across years — inspect, then adapt.

In [10]:
con = duckdb.connect()
con.execute(f"CREATE VIEW trips AS SELECT * FROM read_csv_auto('{sample}', header=True, ignore_errors=true)")
con.execute('DESCRIBE trips').fetchdf()

,column_name,column_type,null,key,default,extra
0,Trip_Id,BIGINT,YES,None,None,None
1,Trip_Duration,BIGINT,YES,None,None,None
2,Start_Station_Id,BIGINT,YES,None,None,None
3,Start_Time,TIMESTAMP,YES,None,None,None
4,Start_Station_Name,VARCHAR,YES,None,None,None
5,End_Station_Id,BIGINT,YES,None,None,None
6,End_Time,TIMESTAMP,YES,None,None,None
7,End_Station_Name,VARCHAR,YES,None,None,None
8,Bike_Id,BIGINT,YES,None,None,None
9,User_Type,VARCHAR,YES,None,None,None


In [11]:
con.execute('SELECT COUNT(*) AS trips FROM trips').fetchdf()

,trips
0,202946


In [12]:
# Auto-detect start/end station columns across schema variants.
cols = [r[0] for r in con.execute('DESCRIBE trips').fetchall()]
def pick(keywords):
    low = [c.lower() for c in cols]
    for i, c in enumerate(low):
        if all(k in c for k in keywords):
            return cols[i]
    return None

start_name = pick(['start', 'station', 'name'])
end_name = pick(['end', 'station', 'name'])
start_id = pick(['start', 'station', 'id'])
end_id = pick(['end', 'station', 'id'])
print({'start_name': start_name, 'end_name': end_name, 'start_id': start_id, 'end_id': end_id})

top = con.execute(f'''
    SELECT "{start_name}" AS origin,
           "{end_name}" AS dest,
           COUNT(*) AS trips
    FROM trips
    WHERE "{start_name}" IS NOT NULL AND "{end_name}" IS NOT NULL
    GROUP BY 1, 2
    ORDER BY trips DESC
    LIMIT 20
''').fetchdf()
top

{'start_name': 'Start_Station_Name', 'end_name': 'End_Station_Name', 'start_id': 'Start_Station_Id', 'end_id': 'End_Station_Id'}


,origin,dest,trips
0,Bay St / College St (East Side),Bay St / College St (East Side),2039
1,Spadina Ave / Harbord St - SMART,Spadina Ave / Harbord St - SMART,1595
2,Bay St / Wellesley St W,Bay St / Wellesley St W,1579
3,College St / Major St,College St / Major St,1364
4,Richmond St E / Yonge St,Richmond St E / Yonge St,1323
5,College St / Huron St,College St / Huron St,1318
6,College Park - Yonge St Entrance,College Park - Yonge St Entrance,1227
7,St. George St / Bloor St W,St. George St / Bloor St W,1225
8,Widmer St / Adelaide St W - SMART,Widmer St / Adelaide St W - SMART,1205
9,King St W / Portland St - SMART,King St W / Portland St - SMART,1148
